<a href="https://colab.research.google.com/github/fajriantomanungki/lomba_kti/blob/main/Data/Koordinat_SPBU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!apt-get update -qq
!apt-get install -y osmium-tool -qq
!pip install geopandas pandas shapely tqdm -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-program-options1.74.0:amd64.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../libboost-program-options1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-program-options1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package osmium-tool.
Preparing to unpack .../osmium-tool_1.14.0-1_amd64.deb ...
Unpacking osmium-tool (1.14.0-1) ...
Setting up libboost-program-options1.74.0:amd64 (1.74.0-14ubuntu3) ...
Setting up osmium-tool (1.14.0-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/l

In [3]:
import os

project_path = "/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua"
data_path = f"{project_path}/data/osm_pbf"
filtered_path = f"{project_path}/data/osm_filtered"
output_path = f"{project_path}/output"

os.makedirs(data_path, exist_ok=True)
os.makedirs(filtered_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

In [4]:
pbf_files = {
    "Sulawesi Barat": "west_sulawesi-latest.osm.pbf",
    "Sulawesi Selatan": "south_sulawesi-latest.osm.pbf",
    "Sulawesi Tengah": "central_sulawesi-latest.osm.pbf",
    "Sulawesi Tenggara": "southeast_sulawesi-latest.osm.pbf",
    "Sulawesi Utara": "north_sulawesi-latest.osm.pbf",
    "Gorontalo": "gorontalo-latest.osm.pbf",
    "Maluku": "maluku-latest.osm.pbf",
    "Maluku Utara": "north_maluku-latest.osm.pbf",
    "Papua": "papua-latest.osm.pbf",
    "Papua Barat": "west_papua-latest.osm.pbf",
    "Papua Selatan": "south_papua-latest.osm.pbf",
    "Papua Tengah": "central_papua-latest.osm.pbf",
    "Papua Pegunungan": "highland_papua-latest.osm.pbf",
    "Papua Barat Daya": "southwest_papua-latest.osm.pbf",
}

base_url = "https://download.openstreetmap.fr/extracts/asia/indonesia"

In [5]:
import requests
from tqdm import tqdm

def download_file(url, out_file):
    if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
        print("Sudah ada:", os.path.basename(out_file))
        return True

    print("Download:", url)

    try:
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))

            with open(out_file, "wb") as f, tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                desc=os.path.basename(out_file)
            ) as bar:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
        return True

    except Exception as e:
        print("Gagal:", os.path.basename(out_file), e)
        return False


download_status = []

for prov, filename in pbf_files.items():
    url = f"{base_url}/{filename}"
    out_file = f"{data_path}/{filename}"

    ok = download_file(url, out_file)
    download_status.append({
        "provinsi": prov,
        "filename": filename,
        "download_ok": ok,
        "path": out_file
    })

Download: https://download.openstreetmap.fr/extracts/asia/indonesia/west_sulawesi-latest.osm.pbf


west_sulawesi-latest.osm.pbf: 100%|██████████| 12.0M/12.0M [00:00<00:00, 61.7MB/s]


Download: https://download.openstreetmap.fr/extracts/asia/indonesia/south_sulawesi-latest.osm.pbf


south_sulawesi-latest.osm.pbf: 100%|██████████| 85.1M/85.1M [00:00<00:00, 91.4MB/s]


Download: https://download.openstreetmap.fr/extracts/asia/indonesia/central_sulawesi-latest.osm.pbf


...

central_papua-latest.osm.pbf: 100%|██████████| 6.92M/6.92M [00:00<00:00, 50.5MB/s]


Download: https://download.openstreetmap.fr/extracts/asia/indonesia/highland_papua-latest.osm.pbf


highland_papua-latest.osm.pbf: 100%|██████████| 8.01M/8.01M [00:00<00:00, 58.7MB/s]


Download: https://download.openstreetmap.fr/extracts/asia/indonesia/southwest_papua-latest.osm.pbf


southwest_papua-latest.osm.pbf: 100%|██████████| 5.79M/5.79M [00:00<00:00, 43.1MB/s]


In [6]:
import subprocess
import os

fuel_pbf_files = []

for prov, filename in pbf_files.items():
    input_pbf = f"{data_path}/{filename}"
    output_pbf = f"{filtered_path}/{filename.replace('.osm.pbf', '_amenity_fuel.osm.pbf')}"

    if not os.path.exists(input_pbf):
        print("Skip, file tidak ada:", prov)
        continue

    cmd = [
        "osmium", "tags-filter",
        input_pbf,
        "n/amenity=fuel",
        "w/amenity=fuel",
        "r/amenity=fuel",
        "-o", output_pbf,
        "--overwrite"
    ]

    print("Filter amenity=fuel:", prov)
    try:
        subprocess.run(cmd, check=True)
        fuel_pbf_files.append({
            "provinsi": prov,
            "path": output_pbf,
            "source_filter": "amenity=fuel"
        })
    except Exception as e:
        print("Gagal filter:", prov, e)

Filter amenity=fuel: Sulawesi Barat
Filter amenity=fuel: Sulawesi Selatan
Filter amenity=fuel: Sulawesi Tengah
Filter amenity=fuel: Sulawesi Tenggara
Filter amenity=fuel: Sulawesi Utara
Filter amenity=fuel: Gorontalo
Filter amenity=fuel: Maluku
Filter amenity=fuel: Maluku Utara
Filter amenity=fuel: Papua
Filter amenity=fuel: Papua Barat
Filter amenity=fuel: Papua Selatan
Filter amenity=fuel: Papua Tengah
Filter amenity=fuel: Papua Pegunungan
Filter amenity=fuel: Papua Barat Daya


In [7]:
geojson_files = []

for item in fuel_pbf_files:
    prov = item["provinsi"]
    input_pbf = item["path"]
    output_geojson = input_pbf.replace(".osm.pbf", ".geojson")

    cmd = [
        "osmium", "export",
        input_pbf,
        "-o", output_geojson,
        "--overwrite"
    ]

    print("Export GeoJSON:", prov)
    try:
        subprocess.run(cmd, check=True)
        geojson_files.append({
            "provinsi": prov,
            "path": output_geojson
        })
    except Exception as e:
        print("Gagal export:", prov, e)

Export GeoJSON: Sulawesi Barat
Export GeoJSON: Sulawesi Selatan
Export GeoJSON: Sulawesi Tengah
Export GeoJSON: Sulawesi Tenggara
Export GeoJSON: Sulawesi Utara
Export GeoJSON: Gorontalo
Export GeoJSON: Maluku
Export GeoJSON: Maluku Utara
Export GeoJSON: Papua
Export GeoJSON: Papua Barat
Export GeoJSON: Papua Selatan
Export GeoJSON: Papua Tengah
Export GeoJSON: Papua Pegunungan
Export GeoJSON: Papua Barat Daya


In [8]:
import geopandas as gpd
import pandas as pd

gdfs = []

for item in geojson_files:
    prov = item["provinsi"]
    path = item["path"]

    print("Baca:", prov)
    gdf = gpd.read_file(path)
    gdf["provinsi_pbf"] = prov
    gdf["source_filter"] = "amenity=fuel"
    gdfs.append(gdf)

spbu_osm = pd.concat(gdfs, ignore_index=True)
spbu_osm = gpd.GeoDataFrame(spbu_osm, geometry="geometry", crs="EPSG:4326")

print("Total objek amenity=fuel:", len(spbu_osm))
spbu_osm.head()

Baca: Sulawesi Barat
Baca: Sulawesi Selatan
Baca: Sulawesi Tengah
Baca: Sulawesi Tenggara
Baca: Sulawesi Utara
Baca: Gorontalo
Baca: Maluku
Baca: Maluku Utara
Baca: Papua
Baca: Papua Barat
Baca: Papua Selatan
Baca: Papua Tengah
Baca: Papua Pegunungan
Baca: Papua Barat Daya
Total objek amenity=fuel: 749


,amenity,brand,brand:wikidata,name,geometry,provinsi_pbf,source_filter,addr:city,addr:housename,addr:housenumber,...,fuel:Bensin,fuel:Pertalite,fuel:Pertamax,fuel:Solar,loc_name,fuel:Dex_Lite,fuel:Premium,note,power,wheelchair
0,fuel,None,None,None,POINT (119.2943 -2.06546),Sulawesi Barat,amenity=fuel,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fuel,None,None,Pom Bensin Tinambung,POINT (119.03353 -3.50202),Sulawesi Barat,amenity=fuel,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,fuel,None,None,Pertamina,POINT (119.14323 -3.4559),Sulawesi Barat,amenity=fuel,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,fuel,Pertamina,Q1641044,Pertamina,POINT (119.22247 -3.39681),Sulawesi Barat,amenity=fuel,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,fuel,Pertamina,Q1641044,Pertamina,POINT (118.86934 -2.68832),Sulawesi Barat,amenity=fuel,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Pastikan geometry sudah berupa titik dan CRS benar
spbu_osm = gpd.GeoDataFrame(spbu_osm, geometry="geometry", crs="EPSG:4326")

# Kalau ada polygon/line, ubah ke centroid agar bisa punya lat lon
spbu_osm_m = spbu_osm.to_crs("EPSG:6933")
spbu_osm_m["geometry"] = spbu_osm_m.geometry.centroid
spbu_osm = spbu_osm_m.to_crs("EPSG:4326")

# Buat koordinat
spbu_osm["lon"] = spbu_osm.geometry.x
spbu_osm["lat"] = spbu_osm.geometry.y

# Pilih kolom yang aman untuk CSV
kolom_prioritas = [
    "id",
    "name",
    "brand",
    "operator",
    "amenity",
    "addr:city",
    "addr:street",
    "provinsi_pbf",
    "source_filter",
    "lat",
    "lon"
]

kolom_ada = [c for c in kolom_prioritas if c in spbu_osm.columns]

spbu_osm_csv = spbu_osm[kolom_ada].copy()

# Simpan CSV
csv_path = f"{output_path}/titik_spbu_osm_amenity_fuel_sulampua.csv"

spbu_osm_csv.to_csv(csv_path, index=False)

print("CSV berhasil disimpan di:")
print(csv_path)

print("Jumlah baris:", len(spbu_osm_csv))
spbu_osm_csv.head()

CSV berhasil disimpan di:
/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/output/titik_spbu_osm_amenity_fuel_sulampua.csv
Jumlah baris: 749


,name,brand,operator,amenity,addr:city,addr:street,provinsi_pbf,source_filter,lat,lon
0,None,None,NaN,fuel,NaN,NaN,Sulawesi Barat,amenity=fuel,-2.065460,119.294299
1,Pom Bensin Tinambung,None,NaN,fuel,NaN,NaN,Sulawesi Barat,amenity=fuel,-3.502024,119.033534
2,Pertamina,None,NaN,fuel,NaN,NaN,Sulawesi Barat,amenity=fuel,-3.455903,119.143231
3,Pertamina,Pertamina,NaN,fuel,NaN,NaN,Sulawesi Barat,amenity=fuel,-3.396806,119.222467
4,Pertamina,Pertamina,NaN,fuel,NaN,NaN,Sulawesi Barat,amenity=fuel,-2.688323,118.869336
